In [ ]:
import sys
from pathlib import Path
import numpy as np
import cmdstanpy

# Add project root to path
sys.path.append(str(Path("..").resolve()))


from helpers import Model, ModelData

fitted_model_folder = Path("../models")

In [ ]:
# Model Variables and Summary Statistics
from constants import PROCESSED_DATA_FOLDER

model_data = ModelData.from_pickle(PROCESSED_DATA_FOLDER / "processed_data.pkl")

# Analyse der Modellvariablen (Summary Statistics)


In [ ]:
########### Summary Statistics ###################
import pandas as pd

summary_data = {"Variable": [], "Mean": [], "SD": []}

# Rating level
summary_data["Variable"].append("Rating")
summary_data["Mean"].append(np.mean(model_data.ratings))
summary_data["SD"].append(np.std(model_data.ratings, ddof=1))

# Berechne Zeitdifferenzen zwischen Bewertungen
temp_diff = np.diff(model_data.days)
temp_diff = temp_diff[temp_diff >= 0]  # Negative Werte entfernen
summary_data["Variable"].append("Days Between Ratings")
summary_data["Mean"].append(np.mean(temp_diff))
summary_data["SD"].append(np.std(temp_diff, ddof=1))

# Sentiment statistics
summary_data["Variable"].append("Sentiment")
summary_data["Mean"].append(np.nanmean(model_data.sentiment))
summary_data["SD"].append(np.nanstd(model_data.sentiment, ddof=1))

# Restaurant Density
summary_data["Variable"].append("Density")
summary_data["Mean"].append(np.mean(model_data.business_covariates["density"]))
summary_data["SD"].append(np.std(model_data.business_covariates["density"], ddof=1))

# Age (in Monaten, daher / 28)
summary_data["Variable"].append("Age (in months)")
summary_data["Mean"].append(np.mean(model_data.business_covariates["Age"]) / 28)
summary_data["SD"].append(np.std(model_data.business_covariates["Age"] / 28, ddof=1))

# Checkin
summary_data["Variable"].append("Check-in rate")
summary_data["Mean"].append(np.mean(model_data.business_covariates["Checkin"]))
summary_data["SD"].append(np.std(model_data.business_covariates["Checkin"], ddof=1))

# Chain status
summary_data["Variable"].append("Chain status")
summary_data["Mean"].append(np.mean(model_data.business_covariates["chain"]))
summary_data["SD"].append(np.std(model_data.business_covariates["chain"], ddof=1))

# ZRI (Rent Level)
summary_data["Variable"].append("Rent level (Zillow Rent Index)")
summary_data["Mean"].append(np.nanmean(model_data.business_covariates["ZRI"]))
summary_data["SD"].append(np.nanstd(model_data.business_covariates["ZRI"], ddof=1))

# Restaurant Size
summary_data["Variable"].append("Restaurant Size (in m^2)")
summary_data["Mean"].append(
    np.nanmean(model_data.business_covariates["Restaurant.Size"])
)
summary_data["SD"].append(
    np.nanstd(model_data.business_covariates["Restaurant.Size"], ddof=1)
)

# Number of Seats
summary_data["Variable"].append("Number of Seats")
summary_data["Mean"].append(
    np.nanmean(model_data.business_covariates["Number.of.Seats"])
)
summary_data["SD"].append(
    np.nanstd(model_data.business_covariates["Number.of.Seats"], ddof=1)
)


# Time
summary_data["Variable"].append("Time")
summary_data["Mean"].append(np.mean(model_data.time))
summary_data["SD"].append(np.std(model_data.time, ddof=1))

# Closed
summary_data["Variable"].append("Closed")
summary_data["Mean"].append(np.mean(model_data.closed))
summary_data["SD"].append(np.std(model_data.closed, ddof=1))

# DataFrame erstellen
summary_df = pd.DataFrame(summary_data)
summary_df

In [ ]:
######## Price Levels ###########

price_levels = {"Price Level": [], "Mean": [], "SD": []}

level_to_label = {1: "<10$", 2: "11-30$", 3: "31-60$", 4: ">60$"}

for lvl in range(1, 4 + 1):
    price_levels["Price Level"].append(f"{lvl} ({level_to_label[lvl]})")
    price_levels["Mean"].append(
        np.mean(model_data.business_covariates["Price.Level"] == lvl)
    )
    price_levels["SD"].append(
        np.std(model_data.business_covariates["Price.Level"] == lvl, ddof=1)
    )

price_df = pd.DataFrame(price_levels)
price_df

In [ ]:
############ Restaurant Categories ################

# Finde alle Kategorie-Spalten (beginnen mit "category_")
category_unique = model_data.business_covariates["category"].unique()
# Für jede einzigartige Kategorie berechnen wir den Mean und SD des Auftretens
category_stats = {"Category": [], "Mean": [], "SD": []}

for cat in category_unique:
    indicator = (model_data.business_covariates["category"] == cat).astype(float)
    category_stats["Category"].append(cat)
    category_stats["Mean"].append(np.mean(indicator))
    category_stats["SD"].append(np.std(indicator, ddof=1))

category_df = pd.DataFrame(category_stats)
category_df

# Laden von trainierten Modellen


In [ ]:
from typing import Literal


def load_fitted_model(model_name: Literal["hmm", "vdhmm"], S: int) -> Model:
    """
    Lädt ein trainiertes CmdStanPy Modell.
    """
    model_path = fitted_model_folder / f"{model_name}_{S}_cmdstan.pkl"
    model = Model.from_pickle(model_path)

    return model

In [ ]:
model_configurations = [
    ("hmm", 2),
    ("hmm", 3),
    ("hmm", 4),
    ("vdhmm", 2),
    ("vdhmm", 3),
    ("vdhmm", 4),
]

n_models = len(model_configurations)

results_df = {
    "LOOIC": [0] * n_models,
    "WAIC": [0] * n_models,
    "LPD_TRAIN": [0] * n_models,
    "LPD_VAL": [0] * n_models,
    "AUC_TRAIN": [0] * n_models,
    "AUC_TEST": [0] * n_models,
}

In [53]:
import arviz as az
from sklearn.metrics import roc_auc_score

# Schleife über alle Modell-Konfigurationen
for k, (model_name, S) in enumerate(model_configurations):
    print(f"Processing {k+1}/{n_models}: {model_name} with {S} states...")

    # Lade das Modell
    model = load_fitted_model(model_name, S)
    log_lik = model.fit.stan_variable("log_lik")
    log_lik_test = model.fit.stan_variable("log_lik_test")
    close_prob = model.fit.stan_variable("close_prob")
    close_prob_mean = close_prob.mean(axis=0)

    idata = az.from_cmdstanpy(model.fit, log_likelihood="log_lik")

    # Berechne looc, waic, lpd
    results_df.loc[k, "LOOIC"] = az.loo(idata, var_name="log_lik").elpd_loo
    results_df.loc[k, "WAIC"] = az.waic(idata, var_name="log_lik").elpd_waic
    results_df.loc[k, "LPD_TRAIN"] = -2 * np.sum(np.log(np.exp(log_lik).mean(axis=0)))
    results_df.loc[k, "LPD_VAL"] = -2 * np.sum(
        np.log(np.exp(log_lik_test).mean(axis=0))
    )

    # Berechne AUC für Trainingsdaten (in Prozent)
    y_true_train = np.array(model.stan_data["Closed"][:500])
    results_df.loc[k, "AUC_TRAIN"] = 100 * roc_auc_score(
        y_true_train, close_prob_mean[:500]
    )

    y_true_test = np.array(model.stan_data["Closed"][500:])
    # Annahme: Die Schrittweite passt, ggf. muss die Dimensionalität geprüft werden!
    close_prob_test_mean = np.mean(model.fit.stan_variable("close_prob"), axis=0)
    results_df.loc[k, "AUC_TEST"] = 100 * roc_auc_score(
        y_true_test, close_prob_test_mean[500:]
    )

results_df

Processing 1/6: hmm with 2 states...


/Users/omidsedighi-mornani/Desktop/STUDIUM WIRTSCHAFTSINFORMATIK/Semester 5/Seminar Gust/SourceCode/.venv/lib/python3.11/site-packages/arviz/stats/diagnostics.py:655: RuntimeWarning: invalid value encountered in scalar subtract
  if (np.max(ary) - np.min(ary)) < np.finfo(float).resolution:  # pylint: disable=no-member
/Users/omidsedighi-mornani/Desktop/STUDIUM WIRTSCHAFTSINFORMATIK/Semester 5/Seminar Gust/SourceCode/.venv/lib/python3.11/site-packages/arviz/stats/stats_utils.py:39: RuntimeWarning: invalid value encountered in subtract
  ary = ary - ary.mean(axis, keepdims=True)
/Users/omidsedighi-mornani/Desktop/STUDIUM WIRTSCHAFTSINFORMATIK/Semester 5/Seminar Gust/SourceCode/.venv/lib/python3.11/site-packages/numpy/_core/_methods.py:190: RuntimeWarning: invalid value encountered in subtract
  x = asanyarray(arr - arrmean)
/Users/omidsedighi-mornani/Desktop/STUDIUM WIRTSCHAFTSINFORMATIK/Semester 5/Seminar Gust/SourceCode/.venv/lib/python3.11/site-packages/arviz/stats/diagnostics.py:655:

Processing 2/6: hmm with 3 states...


/Users/omidsedighi-mornani/Desktop/STUDIUM WIRTSCHAFTSINFORMATIK/Semester 5/Seminar Gust/SourceCode/.venv/lib/python3.11/site-packages/arviz/stats/diagnostics.py:655: RuntimeWarning: invalid value encountered in scalar subtract
  if (np.max(ary) - np.min(ary)) < np.finfo(float).resolution:  # pylint: disable=no-member
/Users/omidsedighi-mornani/Desktop/STUDIUM WIRTSCHAFTSINFORMATIK/Semester 5/Seminar Gust/SourceCode/.venv/lib/python3.11/site-packages/arviz/stats/stats_utils.py:39: RuntimeWarning: invalid value encountered in subtract
  ary = ary - ary.mean(axis, keepdims=True)
/Users/omidsedighi-mornani/Desktop/STUDIUM WIRTSCHAFTSINFORMATIK/Semester 5/Seminar Gust/SourceCode/.venv/lib/python3.11/site-packages/numpy/_core/_methods.py:190: RuntimeWarning: invalid value encountered in subtract
  x = asanyarray(arr - arrmean)
/Users/omidsedighi-mornani/Desktop/STUDIUM WIRTSCHAFTSINFORMATIK/Semester 5/Seminar Gust/SourceCode/.venv/lib/python3.11/site-packages/arviz/stats/diagnostics.py:655:

Processing 3/6: hmm with 4 states...


...

Processing 5/6: vdhmm with 3 states...


/Users/omidsedighi-mornani/Desktop/STUDIUM WIRTSCHAFTSINFORMATIK/Semester 5/Seminar Gust/SourceCode/.venv/lib/python3.11/site-packages/arviz/stats/stats.py:1667: UserWarning: For one or more samples the posterior variance of the log predictive densities exceeds 0.4. This could be indication of WAIC starting to fail. 
See http://arxiv.org/abs/1507.04544 for details
  warnings.warn(


Processing 6/6: vdhmm with 4 states...


/Users/omidsedighi-mornani/Desktop/STUDIUM WIRTSCHAFTSINFORMATIK/Semester 5/Seminar Gust/SourceCode/.venv/lib/python3.11/site-packages/arviz/stats/diagnostics.py:655: RuntimeWarning: invalid value encountered in scalar subtract
  if (np.max(ary) - np.min(ary)) < np.finfo(float).resolution:  # pylint: disable=no-member
/Users/omidsedighi-mornani/Desktop/STUDIUM WIRTSCHAFTSINFORMATIK/Semester 5/Seminar Gust/SourceCode/.venv/lib/python3.11/site-packages/arviz/stats/stats_utils.py:39: RuntimeWarning: invalid value encountered in subtract
  ary = ary - ary.mean(axis, keepdims=True)
/Users/omidsedighi-mornani/Desktop/STUDIUM WIRTSCHAFTSINFORMATIK/Semester 5/Seminar Gust/SourceCode/.venv/lib/python3.11/site-packages/numpy/_core/_methods.py:190: RuntimeWarning: invalid value encountered in subtract
  x = asanyarray(arr - arrmean)
/Users/omidsedighi-mornani/Desktop/STUDIUM WIRTSCHAFTSINFORMATIK/Semester 5/Seminar Gust/SourceCode/.venv/lib/python3.11/site-packages/arviz/stats/diagnostics.py:655:

,Model,LOOIC,WAIC,LPD_TRAIN,LPD_VAL,AUC_TRAIN,AUC_TEST
0,hmm_2,-46377.072577,-46376.953641,92689.309864,85095.826704,79.823828,77.775504
1,hmm_3,-45998.386637,-46074.556595,91377.253982,84031.247946,83.821570,81.359252
2,hmm_4,-45777.128334,-45847.027326,90977.702978,83732.515747,83.485037,80.762935
3,vdhmm_2,-47910.425372,-51638.882236,92175.170923,84720.625888,79.259176,77.290266
4,vdhmm_3,-45756.487995,-45756.290918,91412.271767,84103.533494,85.384529,83.142356
5,vdhmm_4,-45673.501194,-45701.598427,90866.259286,83695.856586,85.655562,83.574978


In [ ]:
import arviz as az
from sklearn.metrics import roc_auc_score

# Schleife über alle Modell-Konfigurationen
for k, (model_name, S) in enumerate(model_configurations):
    print(f"Processing {k+1}/{n_models}: {model_name} with {S} states...")

    # Lade das Modell
    model = load_fitted_model(model_name, S)

    # Extrahiere Samples aus dem CmdStanMCMC fit Objekt
    log_lik = model.fit.stan_variable("log_lik")
    log_lik_test = model.fit.stan_variable("log_lik_test")
    close_prob = model.fit.stan_variable("close_prob")

    # Extrahiere N_train aus stan_data
    N_train = model.stan_data.get("N_train", 500)
    N_total = model.stan_data.get("N_total", 921)

    loo_result = az.loo(log_lik, pointwise=True)
    results_df["LOOIC"][k] = loo_result.loo

    # WAIC berechnen (nur für Trainingsdaten)
    waic_result = az.waic(log_lik, pointwise=True)
    results_df["WAIC"][k] = waic_result.waic

    # LPD_TRAIN berechnen: -2 * sum(log(mean(exp(log_lik))))
    results_df["LPD_TRAIN"][k] = -2 * np.sum(np.log(np.mean(np.exp(log_lik), axis=0)))

    # LPD_VAL berechnen (für Validierungsdaten)
    if N_train < N_total:
        log_lik_val = log_lik[:, N_train:N_total]
        results_df["LPD_VAL"][k] = -2 * np.sum(
            np.log(np.mean(np.exp(log_lik_val), axis=0))
        )

    # AUC berechnen (für Close Probability)

    # Berechne mean über Samples (axis=0)
    close_prob_mean = np.mean(close_prob, axis=0)

    # Hole Closed-Werte aus stan_data
    # R: data_stan$Closed
    y_true = np.array(model.stan_data["Closed"])

    # AUC_TRAIN: erste 500 Beobachtungen
    y_true_train = y_true[0:500]
    y_pred_train = close_prob_mean[0:500]
    results_df["AUC_TRAIN"][k] = 100 * roc_auc_score(y_true_train, y_pred_train)

    # AUC_TEST: val.idx = 501:N_total (in R), entspricht 500:N_total in Python
    # R: data_stan$Closed[val.idx] und close_prob_mean[val.idx]
    val_idx = slice(500, N_total)  # 501:921 in R = 500:921 in Python (0-based)
    y_true_test = y_true[val_idx]
    y_pred_test = close_prob_mean[val_idx]
    results_df["AUC_TEST"][k] = 100 * roc_auc_score(y_true_test, y_pred_test)

print("\nFertig! Erstelle DataFrame...")
# DataFrame erstellen
results_df = pd.DataFrame(results_df)
results_df.insert(0, "Model", [f"{name}_{s}" for name, s in model_configurations])
results_df